# Correlation and multicollinearity analysis

This notebook reproduces the Pearson-correlation and variance inflation factor (VIF) analyses used to support physically guided feature selection.

The experimental dataset is not publicly distributed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

DATA_PATH = Path("../data/BEV_model_ready_dataset.xlsx")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place an authorized copy in the local data/ directory."
    )

df = pd.read_excel(DATA_PATH)

rename_map = {
    "Speed": "v",
    "Acceleration": "a",
    "Road_gradient": "theta",
    "RPM": "MotorSpeed",
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

## 1. Pearson correlation matrix

In [ ]:
variables = [
    "v", "a", "theta",
    "Ew_grade", "Ew_drag", "Ew_roll", "Ew_inertia",
    "APP", "SoC", "BattTemp_max", "MotorSpeed",
    "MotorTorque", "MotorTemp", "E_batt_net"
]

labels = {
    "v": r"$v$",
    "a": r"$a$",
    "theta": r"$\theta$",
    "Ew_grade": r"$E_{w,grade}$",
    "Ew_drag": r"$E_{w,drag}$",
    "Ew_roll": r"$E_{w,roll}$",
    "Ew_inertia": r"$E_{w,inert}$",
    "APP": "APP",
    "SoC": "SoC",
    "BattTemp_max": r"$T_{\mathrm{max},b}$",
    "MotorSpeed": "RPM",
    "MotorTorque": r"$T_e$",
    "MotorTemp": r"$T_m$",
    "E_batt_net": r"$E_{batt,net}$",
}

corr = df[variables].corr(method="pearson")
corr_plot = corr.rename(index=labels, columns=labels)

fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(corr_plot.values, cmap="coolwarm", vmin=-1, vmax=1)

ax.set_xticks(range(len(corr_plot.columns)))
ax.set_yticks(range(len(corr_plot.index)))
ax.set_xticklabels(corr_plot.columns, rotation=45, ha="right", fontsize=14)
ax.set_yticklabels(corr_plot.index, fontsize=14)

for i in range(len(corr_plot.index)):
    for j in range(len(corr_plot.columns)):
        ax.text(
            j, i, f"{corr_plot.iloc[i, j]:.2f}",
            ha="center", va="center", fontsize=10
        )

cbar = fig.colorbar(im, ax=ax, shrink=0.82, pad=0.02)
cbar.set_label("Pearson correlation coefficient", fontsize=14)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "Fig9_correlation_matrix.png", dpi=600, bbox_inches="tight")
plt.show()

## 2. VIF function

In [ ]:
def calculate_vif(dataframe, variables):
    data = dataframe[variables].copy()
    data = data.replace([np.inf, -np.inf], np.nan)

    for col in variables:
        data[col] = pd.to_numeric(data[col], errors="coerce")

    data = data.dropna()

    X = add_constant(data, has_constant="add")
    rows = []

    for i, column in enumerate(X.columns):
        if column == "const":
            continue

        rows.append({
            "Variable": column,
            "VIF": variance_inflation_factor(X.values, i),
        })

    vif_df = pd.DataFrame(rows)
    vif_df["Variable"] = pd.Categorical(
        vif_df["Variable"], categories=variables, ordered=True
    )

    vif_df = vif_df.sort_values("Variable").reset_index(drop=True)
    return vif_df, len(data)

## 3. Initial candidate set

In [ ]:
initial_variables = [
    "v",
    "a",
    "theta",
    "APP",
    "MotorSpeed",
    "BattTemp_max",
    "SoC",
    "MotorTorque",
    "MotorTemp",
    "Ew_drag",
    "Ew_roll",
    "Ew_grade",
    "Ew_inertia",
]

vif_initial, n_initial = calculate_vif(df, initial_variables)

print(f"Complete observations used: {n_initial:,}")
vif_initial

## 4. Final eight-predictor set

In [ ]:
final_variables = [
    "Ew_drag",
    "Ew_roll",
    "Ew_grade",
    "Ew_inertia",
    "MotorTorque",
    "SoC",
    "BattTemp_max",
    "MotorTemp",
]

vif_final, n_final = calculate_vif(df, final_variables)

print(f"Complete observations used: {n_final:,}")
vif_final

### Reference final VIF values reported in Appendix B

- `Ew_drag`: 6.28
- `Ew_roll`: 6.16
- `Ew_grade`: 1.53
- `Ew_inertia`: 1.37
- `MotorTorque`: 1.99
- `SoC`: 3.76
- `BattTemp_max`: 3.40
- `MotorTemp`: 1.16

In [ ]:
vif_initial.to_csv(OUTPUT_DIR / "vif_initial.csv", index=False)
vif_final.to_csv(OUTPUT_DIR / "vif_final.csv", index=False)